In [ ]:
import pickle
import torch
import numpy as np
import pandas as pd
import sys
from tqdm.notebook import tqdm

sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')

from event_log_loader.new_event_log_loader import EventLogDataset

In [ ]:
BASE = '../../../'

ATTACK_DIRS = {
    'last_event_attack': BASE + 'evaluation_results/robustness/helpdesk/last_event_attack_all/robustness_results.pkl',
    'random_event_attack': BASE + 'evaluation_results/robustness/helpdesk/random_event_attack_all/robustness_results.pkl',
    'redo_activity': BASE + 'evaluation_results/robustness/helpdesk/redo_activity/robustness_results.pkl',
    'loop_augmentation': BASE + 'evaluation_results/robustness/helpdesk/loop_augmentation/robustness_results.pkl',
}

ENCODER_PATH = BASE + 'encoded_data/data_encoder/helpdesk_encoder_decoder.pkl'
PROPERTIES_PATH = BASE + 'encoded_data/data_encoder/helpdesk_event_log_properties.pkl'
TRAIN_PATH = BASE + 'encoded_data/helpdesk/helpdesk_all_5_train.pkl'
OUTPUT_PATH = BASE + 'encoded_data/helpdesk/helpdesk_all_5_train_adv_retrain.pkl'

CONCEPT_NAME = 'Activity'
CASE_NAME = 'Case ID'
MIN_SUFFIX_SIZE = 5

## Step 1 & 2: Load robustness results and filter successful adversarial examples

In [ ]:
successful_samples = []
stats = {}

for attack_name, path in ATTACK_DIRS.items():
    print(f'Loading {attack_name} from {path} ...')
    with open(path, 'rb') as f:
        results = pickle.load(f)
    
    count_total = 0
    count_success = 0
    
    for (case_name, prefix_len), entry in results.items():
        count_total += 1
        orig_tuple = entry['original']
        pert_tuple = entry['perturbed']
        
        mean_pred_clean = orig_tuple[2]
        mean_pred_pert = pert_tuple[2]
        
        if mean_pred_clean is None or mean_pred_pert is None:
            continue
        
        clean_activities = [e.get(CONCEPT_NAME) for e in mean_pred_clean]
        pert_activities = [e.get(CONCEPT_NAME) for e in mean_pred_pert]
        
        if clean_activities != pert_activities:
            count_success += 1
            successful_samples.append({
                'attack': attack_name,
                'case_name': case_name,
                'prefix_len': prefix_len,
                'perturbed_prefix': pert_tuple[0],
                'ground_truth_suffix': orig_tuple[1],
            })
    
    stats[attack_name] = {'total': count_total, 'success': count_success}
    print(f'  {attack_name}: {count_success}/{count_total} successful ({100*count_success/count_total:.1f}%)')
    del results

print(f'\nTotal successful adversarial samples: {len(successful_samples)}')
for name, s in stats.items():
    print(f'  {name}: {s["success"]}/{s["total"]}')

## Step 3: Merge perturbed prefix + ground truth suffix into full traces

In [ ]:
encoder_decoder = torch.load(ENCODER_PATH, weights_only=False)
all_categorical_cols = encoder_decoder.categorical_columns
all_continuous_cols = encoder_decoder.continuous_columns + encoder_decoder.continuous_positive_columns
all_feature_cols = all_categorical_cols + all_continuous_cols

print(f'Categorical columns ({len(all_categorical_cols)}): {all_categorical_cols}')
print(f'Continuous columns ({len(all_continuous_cols)}): {all_continuous_cols}')
print(f'Window size: {encoder_decoder.window_size}, Min suffix size: {encoder_decoder.min_suffix_size}')

In [ ]:
trace_dfs = []

for sample in tqdm(successful_samples, desc='Building adversarial traces'):
    prefix_events = sample['perturbed_prefix']
    suffix_events = sample['ground_truth_suffix']
    unique_case_id = f"adv_{sample['attack']}_{sample['case_name']}_{sample['prefix_len']}"
    
    trace_events = list(prefix_events) + list(suffix_events)
    
    eos_event = {col: 'EOS' for col in all_categorical_cols}
    for col in all_continuous_cols:
        eos_event[col] = np.nan
    eos_event[CASE_NAME] = unique_case_id
    
    rows = []
    for event in trace_events:
        row = {CASE_NAME: unique_case_id}
        for col in all_feature_cols:
            row[col] = event.get(col)
        rows.append(row)
    
    for _ in range(MIN_SUFFIX_SIZE):
        rows.append(dict(eos_event))
    
    trace_dfs.append(pd.DataFrame(rows))

adv_df = pd.concat(trace_dfs, ignore_index=True)

for col in all_categorical_cols:
    adv_df[col] = adv_df[col].apply(lambda x: x if pd.isna(x) else str(x))
    adv_df[col] = adv_df[col].astype(object)

for col in all_continuous_cols:
    adv_df[col] = adv_df[col].astype('float32')

print(f'Adversarial DataFrame: {len(adv_df)} rows, {adv_df[CASE_NAME].nunique()} unique traces')
adv_df.head(10)

## Step 4: Encode adversarial traces

In [ ]:
adv_encoded_data, adv_all_categories = encoder_decoder.encode_df(adv_df)

adv_cat_tensors, adv_cont_tensors, adv_case_ids = adv_encoded_data
n_adv_samples = adv_cat_tensors[0].shape[0]
print(f'Encoded adversarial data: {n_adv_samples} training windows')
print(f'Categorical tensor shapes: {[t.shape for t in adv_cat_tensors]}')
print(f'Continuous tensor shapes: {[t.shape for t in adv_cont_tensors]}')

## Step 5: Merge with original training data and save

In [ ]:
train_dataset = torch.load(TRAIN_PATH, weights_only=False)
print(f'Original training set: {len(train_dataset)} samples')

orig_cat_tensors = train_dataset.tensor_list[0]
orig_cont_tensors = train_dataset.tensor_list[1]
orig_case_ids = train_dataset.tensor_list[2]

merged_cat = tuple(
    torch.cat([orig_cat_tensors[i], adv_cat_tensors[i]], dim=0)
    for i in range(len(orig_cat_tensors))
)

merged_cont = tuple(
    torch.cat([orig_cont_tensors[i], adv_cont_tensors[i]], dim=0)
    for i in range(len(orig_cont_tensors))
)

merged_case_ids = tuple(list(orig_case_ids) + list(adv_case_ids))

merged_tensor_tuple = (merged_cat, merged_cont, merged_case_ids)
merged_dataset = EventLogDataset(merged_tensor_tuple, train_dataset.all_categories, encoder_decoder)

print(f'Merged training set: {len(merged_dataset)} samples')
print(f'  Original: {len(train_dataset)}')
print(f'  Adversarial: {n_adv_samples}')

In [ ]:
torch.save(merged_dataset, OUTPUT_PATH)
print(f'Saved merged dataset to {OUTPUT_PATH}')

print('\n--- Summary ---')
print(f'Successful adversarial examples per attack:')
for name, s in stats.items():
    print(f'  {name}: {s["success"]}/{s["total"]} ({100*s["success"]/s["total"]:.1f}%)')
print(f'Total successful adversarial traces: {len(successful_samples)}')
print(f'Total adversarial training windows: {n_adv_samples}')
print(f'Original training samples: {len(train_dataset)}')
print(f'Final merged training samples: {len(merged_dataset)}')